In [14]:
!pip install ultralytics wandb -q

In [15]:
import wandb
from ultralytics import YOLO

In [16]:
wandb.login(key="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")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
import os
import random
import shutil

source_dir = '/content/drive/MyDrive/vehicle_images/'
base_dir = '/content/dataset/'

for split in ['train', 'val', 'test']:
    os.makedirs(f"{base_dir}/{split}/images", exist_ok=True)
    os.makedirs(f"{base_dir}/{split}/labels", exist_ok=True)

valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.JPG', '.BMP')
all_images = [f for f in os.listdir(source_dir) if f.endswith(valid_extensions)]
random.shuffle(all_images)

train_idx = int(len(all_images) * 0.7)
val_idx = int(len(all_images) * 0.9)

splits = {
    'train': all_images[:train_idx],
    'val': all_images[train_idx:val_idx],
    'test': all_images[val_idx:]
}

for split, images in splits.items():
    for img in images:
        shutil.copy(os.path.join(source_dir, img), os.path.join(base_dir, split, 'images', img))

In [23]:
model = YOLO('yolov8n.pt')
coco_to_custom = {2: 0, 1: 1, 5: 2, 7: 3, 3: 4}

for split in ['train', 'val', 'test']:
    img_folder = f"{base_dir}/{split}/images"
    label_folder = f"{base_dir}/{split}/labels"

    for img_name in os.listdir(img_folder):
        img_path = os.path.join(img_folder, img_name)
        results = model.predict(img_path, verbose=False)

        txt_name = os.path.splitext(img_name)[0] + '.txt'
        with open(os.path.join(label_folder, txt_name), 'w') as f:
            for box in results[0].boxes:
                cls_id = int(box.cls[0].item())
                if cls_id in coco_to_custom:
                    new_cls = coco_to_custom[cls_id]
                    x_c, y_c, w, h = box.xywhn[0].tolist()
                    f.write(f"{new_cls} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n")

In [24]:
dataset_info = {
    'train': '/content/dataset/train/images',
    'val': '/content/dataset/val/images',
    'test': '/content/dataset/test/images',
    'nc': 5,
    'names': ['Car', 'Bicycle', 'Bus', 'Truck', 'Motorcycle']
}

In [25]:
import yaml
with open('/content/data.yaml', 'w') as f:
    yaml.dump(dataset_info, f, sort_keys=False)

In [26]:
from ultralytics import YOLO
import wandb

wandb.init(project="YOLOv8_Vehicle_Detection", name="YOLOv8_Small")

model_small = YOLO('yolov8s.pt')

results_small = model_small.train(
    data='/content/data.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    project='Vehicle_Detection',
    name='Model_Small'
)

wandb.finish()

Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Model_Small-3, nbs=64, nms=False, opset=Non

In [27]:
wandb.init(project="YOLOv8_Vehicle_Detection", name="YOLOv8_Medium")
model_medium = YOLO('yolov8m.pt')

results_medium = model_medium.train(
    data='/content/data.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    project='Vehicle_Detection',
    name='Model_Medium'
)
wandb.finish()

Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Model_Medium, nbs=64, nms=False, opset=None

In [28]:
wandb.init(project="YOLOv8_Vehicle_Detection", name="YOLOv8_Large")
model_large = YOLO('yolov8l.pt')

results_large = model_large.train(
    data='/content/data.yaml',
    epochs=30,
    imgsz=640,
    batch=8,
    project='Vehicle_Detection',
    name='Model_Large'
)
wandb.finish()

Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Model_Large, nbs=64, nms=False, opset=None, 